# HE_Hedge (Slave/Follower EA) — Component Testing

This notebook breaks down the `HE_Hedge.mq5` Expert Advisor into testable components.
Each section isolates a logical unit so you can verify it works before deploying to the
MetaTrader 5 terminal via the App Connector.

**EA Role:** Slave / Follower — receives & executes (inverted) trades from HE_Prop master.

**Architecture:**
- ZMQ SUB socket → subscribes to master's PUB events (`EVENT|`, `SNAPSHOT|`)
- ZMQ REP socket → local command channel for Electron app
- License validation → DLL or WebRequest fallback
- Trade copying → with inversion (BUY↔SELL, SL↔TP swap)
- Registration file → `HedgeEdge\<login>.json` in FILE_COMMON for app auto-discovery
- Trade log → `.jsonl` in FILE_COMMON for offline sync

---
## Component 1: Properties & Includes

The EA header declares version, description, and pulls in ZMQ + DLL dependencies.

In [ ]:
# MQL5 Source — Properties & Includes
mql5_properties = r"""
#property copyright "Copyright 2025, HedgEdge Technologies"
#property link      "https://www.hedge-edge.com"
#property version   "3.10"
#property description "HE_slave - HedgEdge Slave/Follower EA"
#property description "Receives & executes trades from HE_prop master."
#property description "Subscribes via ZMQ PUB/SUB (libzmq.dll + libsodium.dll)."
#property strict

//--- Include ZMQ v2 wrapper (CURVE + monitor + topic PUB/SUB)
#include <ZMQv2.mqh>

//--- Windows API for DLL detection
#import "kernel32.dll"
   int GetModuleHandleW(string lpModuleName);
#import

//--- DLL imports for license validation
#import "HedgeEdgeLicense.dll"
   int  ValidateLicense(string key, string account, string broker, string deviceId,
                        string endpointUrl, char &outToken[], char &outError[]);
   int  GetCachedToken(char &outToken[], int tokenLen);
   int  IsTokenValid();
   int  GetTokenTTL();
   void SetEndpoint(string url);
   void ClearCache();
   int  InitializeLibrary();
   void ShutdownLibrary();
#import
"""

print("--- Component 1: Properties & Includes ---")
print("Required files in MQL5/Libraries/:")
print("  - libzmq.dll")
print("  - libsodium.dll")
print("  - HedgeEdgeLicense.dll (optional, graceful fallback)")
print("Required files in MQL5/Include/:")
print("  - ZMQv2.mqh")
print()
print("CHECKPOINT: Verify these files exist in your MT5 installation.")

---
## Component 2: Input Parameters

All user-configurable inputs grouped by category.

In [ ]:
# MQL5 Source — Input Parameters
mql5_inputs = r"""
input group "=== License Settings ==="
input string InpLicenseKey = "";                     // License Key
input string InpDeviceId = "";                       // Device ID (auto if blank)
input string InpEndpointUrl = "https://hedge-edge-app-backend-production.up.railway.app/v1/license/validate";
input int    InpPollIntervalSeconds = 300;           // License Check Interval (s)
input bool   InpDevMode = false;                     // DEV MODE (skip license)

input group "=== Master Connection ==="
input string InpMasterAddress = "localhost";         // Master Address (IP/hostname)
input int    InpMasterDataPort = 51810;              // Master PUB Port (data)
input int    InpMasterCommandPort = 51811;           // Master REP Port (commands)
input bool   InpEnableCurve = false;                 // Enable CURVE Encryption
input string InpMasterPublicKey = "";                // Master Public Key (Z85)

input group "=== Trade Copy Settings ==="
input double InpLotMultiplier = 1.0;                 // Lot Multiplier
input double InpFixedLots = 0.0;                     // Fixed Lot Size (0 = use multiplier)
input double InpMaxLots = 100.0;                     // Maximum Lots per Trade
input int    InpSlippage = 10;                       // Max Slippage (points)
input int    InpMagicNumber = 123456;                // Magic Number
input string InpTradeComment = "HE-Copy";            // Trade Comment Prefix
input bool   InpCopySLTP = true;                     // Copy Stop Loss / Take Profit
input bool   InpInvertTrades = true;                 // Invert Trade Direction (hedge)
input bool   InpCopyCloseSignals = true;             // Copy Close Signals

input group "=== App Communication ==="
input int    InpCommandPort = 51821;                 // Local REP Port (app commands)
input bool   InpEnableLocalCommands = true;          // Enable App Command Channel
"""

print("--- Component 2: Input Parameters ---")
print("Groups:")
print("  1. License Settings    — key, endpoint, poll interval, dev mode")
print("  2. Master Connection   — address, data port 51810, command port 51811, CURVE")
print("  3. Trade Copy Settings — lot sizing, slippage, magic number, inversion")
print("  4. App Communication   — local REP port 51821 for Electron app")
print()
print("KEY CONFIG FOR TESTING:")
print("  - Set InpDevMode = true to skip license during development")
print("  - InpInvertTrades = true is DEFAULT (BUY→SELL, SL↔TP swap)")
print("  - InpMasterAddress = 'localhost' for same-machine testing")

---
## Component 3: Global Variables & Data Structures

Runtime state, ZMQ objects, position mapping, and trade stats.

In [ ]:
# MQL5 Source — Global Variables
mql5_globals = r"""
bool g_isLicenseValid = false;
bool g_isPaused = false;
bool g_dllLoaded = false;
bool g_zmqInitialized = false;
bool g_subscriberConnected = false;
string g_lastError = "";
string g_statusMessage = "Initializing...";
datetime g_lastLicenseCheck = 0;
string g_deviceId = "";

// Runtime-overridable copy settings (pushed by Electron app via SET_CONFIG)
bool   g_invertTrades = true;
double g_lotMultiplier = 1.0;
double g_fixedLots = 0.0;
bool   g_copySLTP = true;

// Stats
ulong g_eventsReceived = 0;
ulong g_tradesCopied = 0;
ulong g_tradesFailed = 0;

// ZMQ sockets
CZmqContext    g_zmqContext;
CZmqSubscriber g_subscriber;     // SUB to master's PUB
CZmqRequester  g_requester;      // REQ to master's REP (unused currently)
CZmqReplier    g_localReplier;   // REP for Electron app commands

// Position mapping (master ticket → slave ticket)
struct PositionMap {
   ulong masterTicket;
   ulong slaveTicket;
   string symbol;
   double volume;
   int    type;
};
PositionMap g_positionMap[];
"""

print("--- Component 3: Global Variables ---")
print("Key runtime state:")
print("  - g_invertTrades, g_lotMultiplier, g_fixedLots, g_copySLTP")
print("    → Can be changed at runtime via SET_CONFIG command from Electron app")
print("  - g_positionMap[] — maps master ticket → slave ticket for close/modify")
print("  - Stats: eventsReceived, tradesCopied, tradesFailed")
print()
print("ZMQ topology:")
print("  SUB ← master PUB (port 51810) — receives trade events")
print("  REP → Electron app (port 51821) — receives PAUSE/RESUME/STATUS/SET_CONFIG")

---
## Component 4: Shared License Key (FILE_COMMON)

Reads license key from `HedgeEdge\license.key` in MT5 Common Files.
The Electron app writes this so all EAs auto-read it.

In [ ]:
# MQL5 Source — ReadSharedLicenseKey()
mql5_read_license = r"""
string ReadSharedLicenseKey()
{
   string filename = "HedgeEdge\\license.key";
   if(!FileIsExist(filename, FILE_COMMON)) return "";
   
   int handle = FileOpen(filename, FILE_READ|FILE_TXT|FILE_COMMON, 0, CP_UTF8);
   if(handle == INVALID_HANDLE) return "";
   
   string key = "";
   if(!FileIsEnding(handle)) key = FileReadString(handle);
   FileClose(handle);
   
   StringTrimLeft(key); StringTrimRight(key);
   if(StringLen(key) < 8) return "";
   return key;
}
"""

# Python equivalent test — verify the Common Files path exists
import os

# MT5 Common Files path (adjust per your installation)
mt5_common = os.path.expandvars(r"%APPDATA%\MetaQuotes\Terminal\Common\Files")
license_path = os.path.join(mt5_common, "HedgeEdge", "license.key")

print("--- Component 4: Shared License Key ---")
print(f"Expected path: {license_path}")
if os.path.exists(license_path):
    with open(license_path, 'r') as f:
        key = f.read().strip()
    print(f"  FOUND — key length: {len(key)} chars")
    print(f"  Preview: {key[:4]}****{key[-4:]}")
else:
    print("  NOT FOUND — Electron app needs to write this file")
    print(f"  Common Files dir exists: {os.path.exists(mt5_common)}")

---
## Component 5: Trade Log Writer

Writes JSONL trade entries for offline sync between EA and Electron app.

In [ ]:
# MQL5 Source — WriteTradeLogEntry()
mql5_trade_log = r"""
void WriteTradeLogEntry(string eventType, string symbol, string side, double volume,
                        double profit, double swap, double commission,
                        ulong masterTicket, ulong slaveTicket,
                        double entryPrice, double closePrice)
{
   string login = IntegerToString(AccountInfoInteger(ACCOUNT_LOGIN));
   string filename = "HedgeEdge\\trade_log_" + login + ".jsonl";
   // Appends one JSON object per line
   // Fields: event, account, broker, masterTicket, slaveTicket, symbol,
   //         side, volume, entryPrice, closePrice, profit, swap, commission,
   //         balance, equity, timestamp, timestampUnix
}
"""

# Python test — check if any trade logs exist
import glob
import json

trade_logs = glob.glob(os.path.join(mt5_common, "HedgeEdge", "trade_log_*.jsonl"))
print("--- Component 5: Trade Log Writer ---")
print(f"Trade log files found: {len(trade_logs)}")
for log_file in trade_logs:
    basename = os.path.basename(log_file)
    with open(log_file, 'r') as f:
        lines = f.readlines()
    print(f"  {basename}: {len(lines)} entries")
    if lines:
        try:
            last_entry = json.loads(lines[-1])
            print(f"    Last: {last_entry.get('event')} {last_entry.get('symbol')} {last_entry.get('side')} @ {last_entry.get('timestamp')}")
        except json.JSONDecodeError:
            print(f"    (last line not valid JSON)")

---
## Component 6: OnInit — Initialization Flow

The full startup sequence: CURVE setup → ZMQ init → Device ID → License → Registration file.

In [ ]:
# MQL5 Source — OnInit() flow
mql5_oninit = r"""
int OnInit()
{
   // 1. CURVE keypair generation (client-side) if InpEnableCurve && InpMasterPublicKey
   // 2. InitializeZMQ() — SUB to master PUB + local REP for Electron
   // 3. Device ID — auto-generate from terminal info if blank
   // 4. License resolution: InpLicenseKey > ReadSharedLicenseKey() > DEV MODE
   // 5. License validation: DLL path or WebRequest fallback
   // 6. WriteRegistrationFile() — HedgeEdge\<login>.json in FILE_COMMON
   // 7. Initialize runtime globals from inputs (g_invertTrades, g_lotMultiplier, etc.)
   return INIT_SUCCEEDED;
}
"""

print("--- Component 6: OnInit Flow ---")
print("Startup sequence:")
print("  1. CURVE setup (optional — client keypair + parse master public key)")
print("  2. ZMQ init (SUB→master:51810, REP on :51821)")
print("  3. Device ID generation")
print("  4. License key resolution (input → shared file → dev mode)")
print("  5. License validation (DLL → WebRequest fallback)")
print("  6. Registration file write")
print("  7. Runtime globals init")
print()
print("FAILURE MODES:")
print("  - ZMQ init fails → INIT_FAILED (missing libzmq.dll)")
print("  - No license key + not dev mode → INIT_PARAMETERS_INCORRECT")
print("  - License validation fails → INIT_FAILED")

---
## Component 7: ZMQ Initialization (Slave Mode)

Creates SUB socket to master PUB + local REP socket for Electron app.

In [ ]:
# MQL5 Source — InitializeZMQ() for Slave
mql5_zmq_init = r"""
bool InitializeZMQ()
{
   // Context
   g_zmqContext.Initialize();
   
   // SUB socket → connect to master PUB
   string dataEndpoint = "tcp://" + InpMasterAddress + ":" + InpMasterDataPort;
   // Subscribe to EVENT| and SNAPSHOT| topics
   g_subscriber.Socket().SetSubscribe("EVENT|");
   g_subscriber.Socket().SetSubscribe("SNAPSHOT|");
   
   // Local REP socket for Electron app commands
   string localEndpoint = "tcp://*:" + InpCommandPort;  // 51821
   g_localReplier.Initialize(g_zmqContext, localEndpoint);
   
   EventSetMillisecondTimer(50);  // 50ms poll rate
   return true;
}
"""

# Python test — verify ZMQ connectivity to expected ports
import socket

ports_to_check = {
    51810: "Master PUB (data events)",
    51811: "Master REP (commands)",
    51821: "Slave local REP (Electron app)",
}

print("--- Component 7: ZMQ Initialization ---")
print("Port availability check (should be FREE before EA starts):")
for port, desc in ports_to_check.items():
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(1)
    result = sock.connect_ex(('127.0.0.1', port))
    status = "IN USE" if result == 0 else "FREE"
    print(f"  :{port} ({desc}) — {status}")
    sock.close()

---
## Component 8: Event Processing — OnTimer / OnTick

The main loop receives master events, processes app commands, checks health & license.

In [ ]:
# MQL5 Source — OnTimer() + OnTick()
mql5_event_loop = r"""
void OnTimer()
{
   if(!g_zmqInitialized) return;
   if(!g_isPaused && g_isLicenseValid)
      ProcessMasterEvents();       // Receive EVENT|/SNAPSHOT| from master
   if(InpEnableLocalCommands)
      ProcessLocalCommands();      // Handle PAUSE/RESUME/STATUS/SET_CONFIG from app
   CheckConnectionHealth();        // Heartbeat timeout detection (15s)
   // License periodic recheck every InpPollIntervalSeconds
}

void OnTick()
{
   // Also processes on tick for lower latency when market is active
   if(!g_zmqInitialized || g_isPaused || !g_isLicenseValid) return;
   ProcessMasterEvents();
}
"""

print("--- Component 8: Event Processing Loop ---")
print("OnTimer (every 50ms):")
print("  1. ProcessMasterEvents() — up to 50 messages per cycle")
print("  2. ProcessLocalCommands() — app commands")
print("  3. CheckConnectionHealth() — heartbeat timeout")
print("  4. License recheck (every 300s default)")
print()
print("OnTick (on every market tick):")
print("  ProcessMasterEvents() — for lower latency during market hours")

---
## Component 9: Trade Event Handlers

Core trade-copying logic: OPENED, CLOSED, MODIFIED, REVERSED.

In [ ]:
# MQL5 Source — HandleEvent() dispatcher + handlers
mql5_event_handlers = r"""
void HandleEvent(string json)
{
   string eventType = ExtractJsonValue(json, "type");
   if(eventType == "POSITION_OPENED")   HandlePositionOpened(json);
   else if(eventType == "POSITION_CLOSED")   HandlePositionClosed(json);
   else if(eventType == "POSITION_MODIFIED") HandlePositionModified(json);
   else if(eventType == "POSITION_REVERSED") HandlePositionReversed(json);
   else if(eventType == "HEARTBEAT")    HandleHeartbeat(json);
   else if(eventType == "CONNECTED")    HandleMasterConnected(json);
   else if(eventType == "DISCONNECTED") HandleMasterDisconnected(json);
   else if(eventType == "ACCOUNT_UPDATE") HandleAccountUpdate(json);
}

// POSITION_OPENED:
//   - Extract symbol, side, volume, SL, TP, masterTicket from data JSON
//   - Duplicate check (masterTicket already mapped → skip)
//   - INVERT: BUY→SELL, swap SL↔TP
//   - CalculateLotSize() → ExecuteOpen()
//   - Store in g_positionMap[]
//   - WriteTradeLogEntry("COPY_OPEN", ...)

// POSITION_CLOSED:
//   - Find slave ticket from g_positionMap
//   - ClosePositionByTicket()
//   - WriteTradeLogEntry("COPY_CLOSE", ...)
//   - Remove from g_positionMap[]

// POSITION_MODIFIED:
//   - Find slave ticket, update SL/TP via ModifyPositionSLTP()

// POSITION_REVERSED:
//   - Delegates to HandlePositionClosed() (close + reopen)
"""

# Python simulation — test trade inversion logic
def simulate_trade_inversion(side, sl, tp, invert=True):
    """Simulate the HE_Hedge trade inversion logic."""
    if invert:
        inverted_side = "SELL" if side == "BUY" else "BUY"
        inverted_sl = tp   # Leader's TP → Follower's SL
        inverted_tp = sl   # Leader's SL → Follower's TP
    else:
        inverted_side = side
        inverted_sl = sl
        inverted_tp = tp
    return inverted_side, inverted_sl, inverted_tp

print("--- Component 9: Trade Event Handlers ---")
print("\nTrade Inversion Tests:")
test_cases = [
    ("BUY",  1.10500, 1.11000),
    ("SELL", 1.11000, 1.10500),
    ("BUY",  0,       1.12000),
    ("SELL", 1.09000, 0),
]
for side, sl, tp in test_cases:
    inv_side, inv_sl, inv_tp = simulate_trade_inversion(side, sl, tp)
    print(f"  Master {side} SL={sl} TP={tp}  →  Slave {inv_side} SL={inv_sl} TP={inv_tp}")

print("\nAll inversions correct? BUY↔SELL and SL↔TP swapped.")

---
## Component 10: Reconciliation Engine

Syncs slave positions with master state on connect and periodically.

In [ ]:
# MQL5 Source — ReconcilePositions()
mql5_reconcile = r"""
void ReconcilePositions(string json)
{
   // 1. Parse positions array from master snapshot
   // 2. For each master position not in g_positionMap:
   //    - Open inverted copy (missed POSITION_OPENED events)
   //    - Add to g_positionMap
   // 3. For each g_positionMap entry where master no longer has it:
   //    - Close orphaned slave position
   //    - Remove from g_positionMap
}
"""

# Python simulation — test reconciliation logic
def reconcile(master_positions, slave_map):
    """Simulate reconciliation between master and slave."""
    actions = []
    master_tickets = {p['ticket'] for p in master_positions}
    mapped_tickets = {m['masterTicket'] for m in slave_map}
    
    # Missing on slave — need to open
    for p in master_positions:
        if p['ticket'] not in mapped_tickets:
            actions.append(f"OPEN: {p['symbol']} {p['side']} (master #{p['ticket']})")
    
    # Orphaned on slave — need to close
    for m in slave_map:
        if m['masterTicket'] not in master_tickets:
            actions.append(f"CLOSE: orphan slave #{m['slaveTicket']} (master #{m['masterTicket']})")
    
    return actions

print("--- Component 10: Reconciliation Engine ---")
master = [
    {"ticket": 100, "symbol": "EURUSD", "side": "BUY"},
    {"ticket": 101, "symbol": "GBPUSD", "side": "SELL"},
    {"ticket": 102, "symbol": "USDJPY", "side": "BUY"},  # NEW — not on slave yet
]
slave = [
    {"masterTicket": 100, "slaveTicket": 200},
    {"masterTicket": 101, "slaveTicket": 201},
    {"masterTicket": 99,  "slaveTicket": 199},  # ORPHAN — master closed it
]

actions = reconcile(master, slave)
for a in actions:
    print(f"  {a}")
print(f"\nActions needed: {len(actions)}")

---
## Component 11: App Command Channel (ProcessLocalCommands)

REP socket handling commands from the Electron app: PAUSE, RESUME, STATUS, SET_CONFIG, OPEN_POSITION, CLOSE_POSITION, etc.

In [ ]:
# MQL5 Source — ProcessLocalCommands()
mql5_local_cmds = r"""
void ProcessLocalCommands()
{
   // Commands handled:
   //   PAUSE          → g_isPaused = true
   //   RESUME         → g_isPaused = false
   //   STATUS         → BuildStatusResponse() (full snapshot)
   //   PING           → pong response with role=slave
   //   CONFIG         → current config (lot multiplier, invert, etc.)
   //   SET_CONFIG     → runtime config push (invertTrades, copySLTP, lotMultiplier, fixedLots)
   //   OPEN_POSITION  → DirectOpenPosition (symbol, side, volume, sl, tp)
   //   MODIFY_POSITION → DirectModifyPosition (ticket, sl, tp)
   //   CLOSE_POSITION → ClosePositionByTicket
   //   CLOSE_ALL      → close all positions, clear g_positionMap
}
"""

# Python test — simulate sending a command to the slave EA
import json as json_mod

commands_to_test = [
    {"action": "PING"},
    {"action": "STATUS"},
    {"action": "SET_CONFIG", "invertTrades": "true", "lotMultiplier": "1.5"},
    {"action": "OPEN_POSITION", "symbol": "EURUSD", "side": "BUY", "volume": "0.01"},
]

print("--- Component 11: App Command Channel ---")
print(f"\nCommands the Electron app can send (port 51821):")
for cmd in commands_to_test:
    print(f"  → {json_mod.dumps(cmd)}")

print("\nTo test live connectivity with ZMQ (requires pyzmq):")
print('  import zmq')
print('  ctx = zmq.Context()')
print('  sock = ctx.socket(zmq.REQ)')
print('  sock.connect("tcp://127.0.0.1:51821")')
print('  sock.send_json({"action": "PING"})')
print('  print(sock.recv_json())')

---
## Component 12: Trade Execution Helpers

`ExecuteOpen()`, `ModifyPositionSLTP()`, `ClosePositionByTicket()`, `CalculateLotSize()`

In [ ]:
# MQL5 Source — Trade Execution
mql5_execution = r"""
ulong ExecuteOpen(string symbol, string side, double lots, double sl, double tp, ulong masterTicket)
{
   // MqlTradeRequest: TRADE_ACTION_DEAL, symbol, volume, deviation, magic, comment
   // Fill mode: IOC > FOK > RETURN (auto-detect from SYMBOL_FILLING_MODE)
   // Returns position ticket (from DEAL_POSITION_ID) or 0 on failure
}

double CalculateLotSize(string symbol, double masterVolume)
{
   // g_fixedLots > 0 ? g_fixedLots : masterVolume * g_lotMultiplier
   // Clamp to SYMBOL_VOLUME_STEP, SYMBOL_VOLUME_MIN, SYMBOL_VOLUME_MAX, InpMaxLots
}

bool ClosePositionByTicket(ulong ticket)
{
   // TRADE_ACTION_DEAL, opposite direction, full volume
}

bool ModifyPositionSLTP(ulong ticket, double sl, double tp)
{
   // TRADE_ACTION_SLTP — only if SL or TP actually changed
}
"""

print("--- Component 12: Trade Execution Helpers ---")

# Lot size calculation simulation
def calculate_lot_size(master_volume, lot_multiplier=1.0, fixed_lots=0.0,
                       max_lots=100.0, vol_min=0.01, vol_max=100.0, vol_step=0.01):
    if fixed_lots > 0:
        lots = fixed_lots
    else:
        lots = master_volume * lot_multiplier
    
    if vol_step > 0:
        lots = int(lots / vol_step) * vol_step
    lots = max(vol_min, min(lots, vol_max, max_lots))
    return round(lots, 2)

print("Lot Size Calculation Tests:")
tests = [
    (1.0, 1.0, 0.0),    # 1:1 copy
    (1.0, 0.5, 0.0),    # Half size
    (1.0, 2.0, 0.0),    # Double
    (1.0, 1.0, 0.05),   # Fixed lots override
    (0.001, 1.0, 0.0),  # Below min
]
for mv, mult, fixed in tests:
    result = calculate_lot_size(mv, mult, fixed)
    print(f"  master={mv}, mult={mult}, fixed={fixed} → {result} lots")

---
## Component 13: License Validation

Two paths: DLL (`HedgeEdgeLicense.dll`) or WebRequest fallback via Railway API.

In [ ]:
# Python test — validate license via the backend API (same as WebRequest fallback)
import requests

ENDPOINT = "https://hedge-edge-app-backend-production.up.railway.app/v1/license/validate"

print("--- Component 13: License Validation ---")
print(f"Endpoint: {ENDPOINT}")
print()

# To test, uncomment and fill in:
# payload = {
#     "licenseKey": "YOUR_KEY_HERE",
#     "accountId": "12345678",
#     "broker": "Your Broker Name",
#     "deviceId": "test-device-notebook"
# }
# resp = requests.post(ENDPOINT, json=payload, timeout=10)
# print(f"HTTP {resp.status_code}: {resp.json()}")

print("MT5 WebRequest requirements:")
print("  1. Tools > Options > Expert Advisors > 'Allow WebRequest for listed URL'")
print("  2. Add: https://hedge-edge-app-backend-production.up.railway.app")
print("  3. If error 4060: URL not whitelisted")
print("  4. If error 4014: WebRequest disabled")

---
## Component 14: Registration File & Dashboard

Registration file for Electron app auto-discovery + on-chart HUD.

In [ ]:
# Python test — check for registration files
import json as json_mod

reg_files = glob.glob(os.path.join(mt5_common, "HedgeEdge", "*.json"))

print("--- Component 14: Registration File & Dashboard ---")
print(f"Registration files found: {len(reg_files)}")
for rf in reg_files:
    basename = os.path.basename(rf)
    try:
        with open(rf, 'r') as f:
            data = json_mod.load(f)
        print(f"  {basename}:")
        print(f"    Role: {data.get('role', 'unknown')}")
        print(f"    Broker: {data.get('broker', 'unknown')}")
        print(f"    Version: {data.get('version', 'unknown')}")
        if data.get('role') == 'slave':
            print(f"    Master: {data.get('masterAddress')}:{data.get('masterDataPort')}")
            print(f"    Command Port: {data.get('commandPort')}")
        elif data.get('role') == 'master':
            print(f"    Data Port: {data.get('dataPort')}")
            print(f"    Command Port: {data.get('commandPort')}")
    except Exception as e:
        print(f"  {basename}: Error reading — {e}")

print("\nDashboard: On-chart HUD shows status, mode (Reverse/Mirror), lot config, copied/failed stats")

---
## Live Test: ZMQ Connectivity to Slave EA

Run this cell **after** attaching HE_Hedge.mq5 to a chart in MT5 with `InpDevMode = true`.

In [ ]:
# Live ZMQ test — requires pyzmq: pip install pyzmq
try:
    import zmq
    import json as json_mod
    
    SLAVE_CMD_PORT = 51821
    
    ctx = zmq.Context()
    sock = ctx.socket(zmq.REQ)
    sock.setsockopt(zmq.RCVTIMEO, 3000)  # 3s timeout
    sock.setsockopt(zmq.SNDTIMEO, 3000)
    sock.connect(f"tcp://127.0.0.1:{SLAVE_CMD_PORT}")
    
    print(f"Connected to slave EA on port {SLAVE_CMD_PORT}")
    
    # PING test
    sock.send_string(json_mod.dumps({"action": "PING"}))
    reply = json_mod.loads(sock.recv_string())
    print(f"PING → {reply}")
    
    # STATUS test
    sock.send_string(json_mod.dumps({"action": "STATUS"}))
    status = json_mod.loads(sock.recv_string())
    print(f"\nSTATUS:")
    print(f"  Account: {status.get('accountId')}")
    print(f"  Broker: {status.get('broker')}")
    print(f"  Balance: {status.get('balance')}")
    print(f"  Licensed: {status.get('isLicenseValid')}")
    print(f"  Paused: {status.get('isPaused')}")
    print(f"  Master Connected: {status.get('masterConnected')}")
    print(f"  Events: {status.get('eventsReceived')} | Copied: {status.get('tradesCopied')} | Failed: {status.get('tradesFailed')}")
    print(f"  Positions: {len(status.get('positions', []))}")
    
    sock.close()
    ctx.term()
    print("\n✓ Slave EA is responding correctly!")
    
except ImportError:
    print("pyzmq not installed. Run: pip install pyzmq")
except zmq.error.Again:
    print("Timeout — Slave EA not responding on port 51821.")
    print("Check: Is HE_Hedge.mq5 attached to a chart? Is InpEnableLocalCommands=true?")
except Exception as e:
    print(f"Error: {e}")

---
## Compilation Checklist

Before compiling with MQL Tools extension:

1. **Files in `MQL5/Libraries/`:**
   - `libzmq.dll`
   - `libsodium.dll`
   - `HedgeEdgeLicense.dll` (optional)

2. **Files in `MQL5/Include/`:**
   - `ZMQv2.mqh`

3. **MT5 Settings:**
   - Tools > Options > Expert Advisors > Allow DLL imports
   - Tools > Options > Expert Advisors > Allow WebRequest for: `https://hedge-edge-app-backend-production.up.railway.app`

4. **EA Parameters for testing:**
   - `InpDevMode = true` (skip license)
   - `InpMasterAddress = localhost`
   - `InpMasterDataPort = 51810`
   - `InpCommandPort = 51821`
   - `InpInvertTrades = true`

---
# DLL Auto-Loading: Eliminating Manual Library Placement

**Goal:** Clients download `.ex5` files, attach to charts, and the EAs find `libzmq.dll` + `libsodium.dll` automatically — no manual copy to `MQL5\Libraries\`.

## Three Strategies (tested below)

| # | Strategy | How it works | Complexity |
|---|----------|-------------|------------|
| 1 | **`SetDllDirectoryW` in OnInit** | EA calls Windows API to add Common Files to DLL search path before first ZMQ call | Low — MQL5 code change only |
| 2 | **Electron app auto-deploy** | App detects MT5 installs and copies DLLs to each terminal's `Libraries\` folder | Medium — app-side logic |
| 3 | **Monolithic DLL** | Statically link libzmq + libsodium into one `HedgeEdge.dll` | High — requires C++ build |

**Recommended: Strategy 1 + 2 combined.** The Electron app deploys DLLs to Common Files on install, and the EA uses `SetDllDirectoryW` to find them there. This gives true zero-touch for clients.

---
## Strategy 1: `SetDllDirectoryW` — Add DLL Search Path

**How MQL5 loads DLLs:** `#import "libzmq.dll"` triggers *lazy loading* — the DLL is loaded on the first function call, NOT at `#import` time. This means we can call `SetDllDirectoryW` in `OnInit()` **before** any ZMQ function to redirect where Windows searches.

**The code change in the EA:**
```mql5
#import "kernel32.dll"
   int  GetModuleHandleW(string lpModuleName);
   int  SetDllDirectoryW(string lpPathName);    // <-- ADD THIS
   int  AddDllDirectory(string newDirectory);    // <-- Win8+ alternative
#import
```

Then at the very start of `OnInit()`:
```mql5
int OnInit()
{
   // Point Windows DLL search to Common Files where Electron deploys libs
   string commonPath = TerminalInfoString(TERMINAL_COMMONDATA_PATH) + "\\Files\\HedgeEdge\\lib";
   SetDllDirectoryW(commonPath);
   Print("DLL search path set to: ", commonPath);
   
   // Now any #import "libzmq.dll" call will also search in commonPath
   // ... rest of OnInit ...
}
```

**DLL placement by Electron app:**
```
%APPDATA%\MetaQuotes\Terminal\Common\Files\HedgeEdge\lib\
    ├── libzmq.dll
    ├── libsodium.dll
    └── HedgeEdgeLicense.dll (optional)
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# TEST: Strategy 1 — Verify SetDllDirectoryW feasibility
# ═══════════════════════════════════════════════════════════════════
import os
import ctypes
import ctypes.wintypes

print("=== Strategy 1: SetDllDirectoryW Test ===\n")

# 1. Locate MT5 Common Files path
mt5_common = os.path.expandvars(r"%APPDATA%\MetaQuotes\Terminal\Common\Files")
lib_path = os.path.join(mt5_common, "HedgeEdge", "lib")

print(f"Target DLL directory: {lib_path}")
print(f"  Exists: {os.path.exists(lib_path)}")

if not os.path.exists(lib_path):
    os.makedirs(lib_path, exist_ok=True)
    print(f"  Created: {lib_path}")

# 2. Check if DLLs are already there
dlls_needed = ["libzmq.dll", "libsodium.dll"]
dll_status = {}
for dll in dlls_needed:
    full = os.path.join(lib_path, dll)
    exists = os.path.exists(full)
    dll_status[dll] = exists
    size = os.path.getsize(full) if exists else 0
    print(f"  {dll}: {'FOUND' if exists else 'MISSING'}" + (f" ({size:,} bytes)" if exists else ""))

# 3. Test SetDllDirectoryW itself (Windows API)
kernel32 = ctypes.WinDLL("kernel32", use_last_error=True)
SetDllDirectoryW = kernel32.SetDllDirectoryW
SetDllDirectoryW.argtypes = [ctypes.wintypes.LPCWSTR]
SetDllDirectoryW.restype = ctypes.wintypes.BOOL

result = SetDllDirectoryW(lib_path)
print(f"\nSetDllDirectoryW('{lib_path}')")
print(f"  Result: {'SUCCESS' if result else 'FAILED'}")
if not result:
    print(f"  Error: {ctypes.get_last_error()}")

# 4. Test if we can load DLLs from there
print("\nDLL Loading test (from new search path):")
for dll in dlls_needed:
    full = os.path.join(lib_path, dll)
    if os.path.exists(full):
        try:
            handle = ctypes.WinDLL(full)
            print(f"  {dll}: LOADED successfully")
        except OSError as e:
            print(f"  {dll}: LOAD FAILED — {e}")
    else:
        print(f"  {dll}: SKIPPED — not in lib folder yet")

# 5. Reset DLL directory
SetDllDirectoryW(None)

print("\n" + "="*60)
print("VERDICT: SetDllDirectoryW works from Python (same Windows API")
print("as MQL5 uses). If DLLs are placed in the lib folder above,")
print("the EA can find them without manual copy to MQL5\\Libraries\\.")

---
## Strategy 2: Electron App Auto-Deploy

The Electron app already writes to `Common\Files\HedgeEdge\` (license key, registration files). We extend this to also deploy DLLs.

**Auto-deploy targets:**

1. **Common Files** (`SetDllDirectoryW` path — shared across all terminals):
   ```
   %APPDATA%\MetaQuotes\Terminal\Common\Files\HedgeEdge\lib\
   ```

2. **Per-terminal Libraries** (direct — no `SetDllDirectoryW` needed):
   ```
   %APPDATA%\MetaQuotes\Terminal\<TERMINAL_ID>\MQL5\Libraries\
   ```
   The app can discover terminal IDs by scanning `%APPDATA%\MetaQuotes\Terminal\`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# TEST: Strategy 2 — Discover MT5 terminals & their Libraries paths
# ═══════════════════════════════════════════════════════════════════
import os
import glob

print("=== Strategy 2: MT5 Terminal Discovery ===\n")

mt5_base = os.path.expandvars(r"%APPDATA%\MetaQuotes\Terminal")

# 1. Scan for terminal data directories
if os.path.exists(mt5_base):
    entries = os.listdir(mt5_base)
    terminal_ids = [e for e in entries if len(e) == 32 and e.isalnum()]
    
    print(f"MT5 Terminal base: {mt5_base}")
    print(f"Terminal instances found: {len(terminal_ids)}\n")
    
    for tid in terminal_ids:
        lib_dir = os.path.join(mt5_base, tid, "MQL5", "Libraries")
        include_dir = os.path.join(mt5_base, tid, "MQL5", "Include")
        experts_dir = os.path.join(mt5_base, tid, "MQL5", "Experts")
        
        print(f"Terminal: {tid[:8]}...")
        print(f"  Libraries: {lib_dir}")
        print(f"    Exists: {os.path.exists(lib_dir)}")
        
        if os.path.exists(lib_dir):
            dlls = [f for f in os.listdir(lib_dir) if f.lower().endswith('.dll')]
            print(f"    DLLs present: {dlls if dlls else '(none)'}")
            has_zmq = 'libzmq.dll' in [d.lower() for d in dlls]
            has_sodium = 'libsodium.dll' in [d.lower() for d in dlls]
            print(f"    libzmq.dll: {'YES' if has_zmq else 'MISSING'}")
            print(f"    libsodium.dll: {'YES' if has_sodium else 'MISSING'}")
        
        if os.path.exists(include_dir):
            mqhs = [f for f in os.listdir(include_dir) if f.lower().endswith('.mqh')]
            zmq_mqh = [f for f in mqhs if 'zmq' in f.lower()]
            print(f"    ZMQ includes: {zmq_mqh if zmq_mqh else '(none)'}")
        
        if os.path.exists(experts_dir):
            ex5s = glob.glob(os.path.join(experts_dir, "**", "*.ex5"), recursive=True)
            he_files = [os.path.basename(f) for f in ex5s if 'HE_' in os.path.basename(f) or 'hedge' in os.path.basename(f).lower()]
            if he_files:
                print(f"    HedgeEdge EAs: {he_files}")
        print()
else:
    print(f"MT5 not found at: {mt5_base}")

# 2. Show Common Files path
common_files = os.path.join(mt5_base, "Common", "Files")
print(f"Common Files: {common_files}")
print(f"  Exists: {os.path.exists(common_files)}")
he_common = os.path.join(common_files, "HedgeEdge")
if os.path.exists(he_common):
    files = os.listdir(he_common)
    print(f"  HedgeEdge folder contents: {files}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# ACTION: Deploy DLLs from source to Common Files + all terminals
# ═══════════════════════════════════════════════════════════════════
# Run this cell to simulate what the Electron app installer would do.
# It copies DLLs from your dev folder to ALL detected MT5 locations.
import os
import shutil

print("=== DLL Auto-Deploy Simulation ===\n")

# Source: where DLLs currently live (adjust if different)
# Check multiple possible source locations
possible_sources = [
    # Alongside the .mq5 files
    os.path.dirname(os.path.abspath(".")),
    # Common developer locations
    os.path.expandvars(r"%APPDATA%\MetaQuotes\Terminal\Common\Files\HedgeEdge\lib"),
]

# Also check all MT5 terminal Libraries folders for existing DLLs to use as source
mt5_base = os.path.expandvars(r"%APPDATA%\MetaQuotes\Terminal")
if os.path.exists(mt5_base):
    entries = os.listdir(mt5_base)
    for e in entries:
        if len(e) == 32 and e.isalnum():
            possible_sources.append(os.path.join(mt5_base, e, "MQL5", "Libraries"))

# Find a source folder that has both DLLs
dll_source = None
for src in possible_sources:
    if os.path.exists(src):
        has_zmq = os.path.exists(os.path.join(src, "libzmq.dll"))
        has_sodium = os.path.exists(os.path.join(src, "libsodium.dll"))
        if has_zmq and has_sodium:
            dll_source = src
            break

if dll_source:
    print(f"Source found: {dll_source}")
    zmq_size = os.path.getsize(os.path.join(dll_source, "libzmq.dll"))
    sodium_size = os.path.getsize(os.path.join(dll_source, "libsodium.dll"))
    print(f"  libzmq.dll: {zmq_size:,} bytes")
    print(f"  libsodium.dll: {sodium_size:,} bytes\n")
    
    # Deploy targets
    targets = []
    
    # Target 1: Common Files (for SetDllDirectoryW strategy)
    common_lib = os.path.join(mt5_base, "Common", "Files", "HedgeEdge", "lib")
    targets.append(("Common Files", common_lib))
    
    # Target 2: Each terminal's Libraries folder
    if os.path.exists(mt5_base):
        for e in os.listdir(mt5_base):
            if len(e) == 32 and e.isalnum():
                lib_dir = os.path.join(mt5_base, e, "MQL5", "Libraries")
                if os.path.exists(lib_dir):
                    targets.append((f"Terminal {e[:8]}...", lib_dir))
    
    # NOTE: This is a DRY RUN — uncomment the shutil lines to actually copy
    print("Deploy plan:")
    for name, target in targets:
        print(f"\n  → {name}: {target}")
        os.makedirs(target, exist_ok=True)
        for dll in ["libzmq.dll", "libsodium.dll"]:
            src_file = os.path.join(dll_source, dll)
            dst_file = os.path.join(target, dll)
            already = os.path.exists(dst_file)
            same_size = already and os.path.getsize(dst_file) == os.path.getsize(src_file)
            if same_size:
                print(f"    {dll}: already up-to-date")
            elif already:
                print(f"    {dll}: EXISTS but different size — would OVERWRITE")
                # shutil.copy2(src_file, dst_file)  # UNCOMMENT TO DEPLOY
            else:
                print(f"    {dll}: MISSING — would COPY")
                # shutil.copy2(src_file, dst_file)  # UNCOMMENT TO DEPLOY
    
    print("\n" + "="*60)
    print("DRY RUN complete. Uncomment shutil.copy2 lines above to deploy.")
    print("The Electron app will do this automatically in production.")
else:
    print("No source DLLs found in any known location.")
    print("Checked:")
    for src in possible_sources:
        print(f"  {src}: {'exists' if os.path.exists(src) else 'not found'}")

## Strategy 3 — Monolithic DLL (Reference Only)

**Concept**: Statically link `libzmq` + `libsodium` into a single `HedgeEdge.dll`, so clients only need one file.

**Pros**: Single DLL, no path manipulation needed  
**Cons**: Requires C++ build toolchain (CMake + MSVC), must rebuild on every upstream version bump, must re-export 40+ ZMQ functions with identical signatures

**Verdict**: High complexity, not recommended for initial release.

---

## ✅ Revised Production Plan (Direct Deploy + Fallback)

The Electron app already scans `%APPDATA%\MetaQuotes\Terminal\` for GUID folders via `terminal-detector.ts`. We leverage this to **copy DLLs directly into each terminal's `MQL5\Libraries\`** — the native location MT5 checks by default.

| Phase | Action | Component |
|-------|--------|-----------|
| **App Start** | `dll-deployer.ts` scans all GUID folders under `MetaQuotes\Terminal\` | Electron main process |
| **Direct Deploy** | Copies `libzmq.dll` + `libsodium.dll` from bundled `agents/mt5/lib/` → each terminal's `MQL5\Libraries\` | `deployDllsToTerminals()` |
| **Common Fallback** | Also copies to `Common\Files\HedgeEdge\lib\` | Same function |
| **EA OnInit** | `SetupDllSearchPath()` checks Libraries first (priority 1), then tries `SetDllDirectoryW` pointing to Common path (priority 2) | MQL5 EA |
| **DLL Load** | Windows finds DLLs in `Libraries\` natively — no path tricks needed | OS (automatic) |

**Result**: Zero manual steps for the client — install app, attach EAs, done.

### EA Priority Logic
```
1. Is libzmq already loaded? → done (return)
2. Does MQL5\Libraries\libzmq.dll exist? → done (MT5 finds it natively)
3. SetDllDirectoryW(Common\Files\HedgeEdge\lib) → fallback if app hasn't deployed yet
```

## MQL5 Code Patch — Revised SetupDllSearchPath (v2)

Both `HE_Hedge.mq5` and `HE_Prop.mq5` have been patched with the priority-based approach:

### kernel32.dll import block (already applied)
```mql5
#import "kernel32.dll"
   int  GetModuleHandleW(string lpModuleName);
   bool SetDllDirectoryW(string lpPathName);
#import
```

### SetupDllSearchPath() — priority-based (already applied)
```mql5
void SetupDllSearchPath()
{
   // Priority 1: Check if DLL already loaded in process
   if(GetModuleHandleW("libzmq") != 0)
   {
      Print("[DLL] libzmq.dll already loaded in process");
      return;
   }
   
   // Priority 2: Check if Electron deployed DLLs to Libraries (native path)
   if(FileIsExist("..\\Libraries\\libzmq.dll", 0))
   {
      Print("[DLL] Found libzmq.dll in MQL5\\Libraries\\ (direct deploy)");
      return;  // MT5 finds them natively — no path tricks needed
   }
   
   // Priority 3: SetDllDirectoryW fallback to Common path
   string commonPath = TerminalInfoString(TERMINAL_COMMONDATA_PATH);
   string dllPath    = commonPath + "\\Files\\HedgeEdge\\lib";
   
   if(SetDllDirectoryW(dllPath))
   {
      Print("[DLL] Search path set: ", dllPath);
      if(!FileIsExist("HedgeEdge\\lib\\libzmq.dll", FILE_COMMON))
         Print("[DLL] WARNING: libzmq.dll not found — install Hedge Edge app first");
   }
   else
   {
      Print("[DLL] WARNING: SetDllDirectoryW failed — copy DLLs to MQL5\\Libraries\\ manually");
   }
}
```

### Electron-side: `dll-deployer.ts` (new file)
```
electron/dll-deployer.ts → deployDllsToTerminals()
  ├── Copies from agents/mt5/lib/ to each terminal's MQL5/Libraries/
  ├── Copies to Common/Files/HedgeEdge/lib/ (SetDllDirectoryW fallback)
  └── Called in main.ts before ZMQ auto-reconnect
```

Run the next cell to validate all patches.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# VALIDATE: Preview the exact MQL5 code patches for both EAs
# ═══════════════════════════════════════════════════════════════════
import os, re

print("=== MQL5 Patch Validation ===\n")

# Locate both .mq5 files
base = os.path.dirname(os.path.abspath("__file__"))
mq5_files = {}
for root, dirs, files in os.walk(os.path.join(base, "..")):
    for f in files:
        if f in ("HE_Hedge.mq5", "HE_Prop.mq5"):
            mq5_files[f] = os.path.join(root, f)

if not mq5_files:
    # Try broader search
    search_root = os.path.expandvars(r"%USERPROFILE%\Desktop\Business")
    for root, dirs, files in os.walk(search_root):
        for f in files:
            if f in ("HE_Hedge.mq5", "HE_Prop.mq5"):
                mq5_files[f] = os.path.join(root, f)

for name, path in sorted(mq5_files.items()):
    print(f"\n{'='*60}")
    print(f"FILE: {name}")
    print(f"PATH: {path}")
    print(f"{'='*60}")
    
    with open(path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Check 1: Does it already have SetDllDirectoryW?
    if 'SetDllDirectoryW' in content:
        print("  ✅ SetDllDirectoryW already present!")
        continue
    
    # Check 2: Find the kernel32.dll import block
    k32_match = re.search(r'#import\s+"kernel32\.dll".*?#import', content, re.DOTALL)
    if k32_match:
        print(f"  Found kernel32.dll import block (chars {k32_match.start()}-{k32_match.end()}):")
        print(f"  Current:\n    {k32_match.group().replace(chr(10), chr(10) + '    ')}")
        print(f"\n  PATCH: Add 'bool SetDllDirectoryW(string lpPathName);' before closing #import")
    else:
        print("  ⚠ No kernel32.dll import block found — need to add one")
    
    # Check 3: Find OnInit
    oninit_match = re.search(r'int\s+OnInit\(\)', content)
    if oninit_match:
        # Show the first few lines after OnInit
        start = oninit_match.start()
        snippet = content[start:start+200]
        print(f"\n  Found OnInit() at char {start}:")
        print(f"    {snippet[:150].replace(chr(10), chr(10) + '    ')}...")
        print(f"\n  PATCH: Insert 'SetupDllSearchPath();' as first statement in OnInit body")
    
    # Check 4: Check current DLL imports
    dll_imports = re.findall(r'#import\s+"([^"]+\.dll)"', content)
    print(f"\n  DLL imports found: {dll_imports}")
    
    print(f"\n  VERDICT: {'2 patches needed' if not 'SetDllDirectoryW' in content else 'Already patched'}")

print("\n\nReady to apply patches. Run the next cell to modify the actual .mq5 files.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# APPLY PATCH: Add SetDllDirectoryW to both HE_Hedge.mq5 & HE_Prop.mq5
# ═══════════════════════════════════════════════════════════════════
# This cell modifies the actual .mq5 source files.
# A .bak backup is created before any changes.
import os, re, shutil

print("=== Applying SetDllDirectoryW Patch ===\n")

# Locate .mq5 files
base = os.path.dirname(os.path.abspath("__file__"))
mq5_files = {}
for root, dirs, files in os.walk(os.path.join(base, "..")):
    for f in files:
        if f in ("HE_Hedge.mq5", "HE_Prop.mq5"):
            mq5_files[f] = os.path.join(root, f)

if not mq5_files:
    search_root = os.path.expandvars(r"%USERPROFILE%\Desktop\Business")
    for root, dirs, files in os.walk(search_root):
        for f in files:
            if f in ("HE_Hedge.mq5", "HE_Prop.mq5"):
                mq5_files[f] = os.path.join(root, f)

HELPER_FUNC = '''
//+------------------------------------------------------------------+
//| Auto-configure DLL search path (zero-touch DLL deployment)       |
//| Electron installer places DLLs in Common\\Files\\HedgeEdge\\lib   |
//+------------------------------------------------------------------+
void SetupDllSearchPath()
{
   string commonPath = TerminalInfoString(TERMINAL_COMMONDATA_PATH);
   string dllPath    = commonPath + "\\\\Files\\\\HedgeEdge\\\\lib";
   
   if(SetDllDirectoryW(dllPath))
      Print("[DLL] Search path set: ", dllPath);
   else
   {
      Print("[DLL] WARNING: SetDllDirectoryW failed for: ", dllPath);
      Print("[DLL] Falling back to MQL5\\\\Libraries\\\\ (manual install required)");
   }
}
'''

for name, path in sorted(mq5_files.items()):
    print(f"\nProcessing: {name}")
    print(f"  Path: {path}")
    
    with open(path, 'r', encoding='utf-8') as f:
        content = f.read()
    original = content
    
    if 'SetDllDirectoryW' in content:
        print("  ⏭ Already patched — skipping")
        continue
    
    # Backup
    bak_path = path + '.bak'
    shutil.copy2(path, bak_path)
    print(f"  📋 Backup: {bak_path}")
    
    changes = 0
    
    # PATCH 1: Add SetDllDirectoryW to kernel32.dll import block
    k32_pattern = r'(#import\s+"kernel32\.dll"\s*\n)(.*?)(#import)'
    k32_match = re.search(k32_pattern, content, re.DOTALL)
    if k32_match:
        # Insert before the closing #import
        insert_line = "   bool SetDllDirectoryW(string lpPathName);\n"
        replacement = k32_match.group(1) + k32_match.group(2) + insert_line + k32_match.group(3)
        content = content[:k32_match.start()] + replacement + content[k32_match.end():]
        print("  ✅ Added SetDllDirectoryW to kernel32.dll import block")
        changes += 1
    else:
        print("  ⚠ kernel32.dll import block not found — adding new block")
        # Add before the first #import
        first_import = content.find('#import')
        if first_import >= 0:
            new_block = '#import "kernel32.dll"\n   int  GetModuleHandleW(string path);\n   bool SetDllDirectoryW(string lpPathName);\n#import\n\n'
            content = content[:first_import] + new_block + content[first_import:]
            changes += 1
    
    # PATCH 2: Add SetupDllSearchPath helper function before OnInit
    oninit_match = re.search(r'\n(int\s+OnInit\(\))', content)
    if oninit_match:
        content = content[:oninit_match.start()] + HELPER_FUNC + '\n' + content[oninit_match.start():]
        print("  ✅ Added SetupDllSearchPath() helper function")
        changes += 1
    
    # PATCH 3: Call SetupDllSearchPath() at start of OnInit
    # Find the opening brace of OnInit
    oninit_body = re.search(r'int\s+OnInit\(\)\s*\n\{', content)
    if oninit_body:
        brace_end = oninit_body.end()
        insert_call = "\n   SetupDllSearchPath();   // Auto-configure DLL search path\n"
        content = content[:brace_end] + insert_call + content[brace_end:]
        print("  ✅ Added SetupDllSearchPath() call at start of OnInit()")
        changes += 1
    
    if changes > 0:
        with open(path, 'w', encoding='utf-8') as f:
            f.write(content)
        print(f"  💾 Saved with {changes} patches applied")
    else:
        print(f"  ⚠ No changes made")

print("\n" + "="*60)
print("Patch complete. Compile both EAs in MetaEditor to verify.")
print("Restore from .bak files if needed.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# END-TO-END VERIFICATION: Confirm patches applied correctly
# ═══════════════════════════════════════════════════════════════════
import os, re

print("=== Post-Patch Verification ===\n")

base = os.path.dirname(os.path.abspath("__file__"))
mq5_files = {}
for root, dirs, files in os.walk(os.path.join(base, "..")):
    for f in files:
        if f in ("HE_Hedge.mq5", "HE_Prop.mq5"):
            mq5_files[f] = os.path.join(root, f)

if not mq5_files:
    search_root = os.path.expandvars(r"%USERPROFILE%\Desktop\Business")
    for root, dirs, files in os.walk(search_root):
        for f in files:
            if f in ("HE_Hedge.mq5", "HE_Prop.mq5"):
                mq5_files[f] = os.path.join(root, f)

all_pass = True
for name, path in sorted(mq5_files.items()):
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    
    with open(path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    checks = {
        "SetDllDirectoryW declared in kernel32 import": bool(re.search(r'#import\s+"kernel32\.dll".*?SetDllDirectoryW.*?#import', content, re.DOTALL)),
        "SetupDllSearchPath() function defined": 'void SetupDllSearchPath()' in content,
        "SetupDllSearchPath() called in OnInit": bool(re.search(r'int\s+OnInit\(\).*?SetupDllSearchPath\(\)', content, re.DOTALL)),
        "Uses TERMINAL_COMMONDATA_PATH": 'TERMINAL_COMMONDATA_PATH' in content,
        "DLL path includes HedgeEdge/lib": 'HedgeEdge' in content and ('lib' in content),
    }
    
    for check, passed in checks.items():
        status = "✅" if passed else "❌"
        if not passed:
            all_pass = False
        print(f"  {status} {check}")
    
    # Show the patched kernel32 block
    k32 = re.search(r'(#import\s+"kernel32\.dll".*?#import)', content, re.DOTALL)
    if k32:
        print(f"\n  kernel32.dll block:")
        for line in k32.group().split('\n'):
            print(f"    {line}")

print(f"\n{'='*50}")
print(f"  Overall: {'ALL CHECKS PASSED ✅' if all_pass else 'SOME CHECKS FAILED ❌'}")
print(f"{'='*50}")
if all_pass:
    print("\nNext steps:")
    print("  1. Open MetaEditor and compile both EAs")
    print("  2. Ensure Electron app installer copies DLLs to:")
    print("     %APPDATA%\\MetaQuotes\\Terminal\\Common\\Files\\HedgeEdge\\lib\\")
    print("  3. Attach EAs to charts — DLLs will load automatically")
    print("  4. Check Experts tab for '[DLL] Search path set: ...' message")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SIMULATE: What the Electron dll-deployer.ts will do at app startup
# ═══════════════════════════════════════════════════════════════════
# This simulates the exact logic from electron/dll-deployer.ts
import os, shutil

print("=== Electron DLL Deployer Simulation ===\n")

REQUIRED_DLLS = ["libzmq.dll", "libsodium.dll"]
MT5_BASE = os.path.expandvars(r"%APPDATA%\MetaQuotes\Terminal")
COMMON_LIB = os.path.join(MT5_BASE, "Common", "Files", "HedgeEdge", "lib")

# Step 1: Find bundled DLLs (in production: agents/mt5/lib/)
# For this test, any existing copy serves as the source
dll_source = None
if os.path.exists(MT5_BASE):
    for entry in os.listdir(MT5_BASE):
        if len(entry) == 32 and entry.isalnum():
            lib_dir = os.path.join(MT5_BASE, entry, "MQL5", "Libraries")
            if all(os.path.exists(os.path.join(lib_dir, d)) for d in REQUIRED_DLLS):
                dll_source = lib_dir
                break

if not dll_source:
    print("⚠ No source DLLs found anywhere. Place libzmq.dll and libsodium.dll")
    print("  in agents/mt5/lib/ within the Electron project to bundle them.")
else:
    print(f"Source: {dll_source}")
    for d in REQUIRED_DLLS:
        sz = os.path.getsize(os.path.join(dll_source, d))
        print(f"  {d}: {sz:,} bytes")
    
    deployed = 0
    skipped = 0
    
    # Step 2: Deploy to Common path (SetDllDirectoryW fallback)
    print(f"\n--- Common Files fallback ---")
    print(f"  Target: {COMMON_LIB}")
    os.makedirs(COMMON_LIB, exist_ok=True)
    for dll in REQUIRED_DLLS:
        src = os.path.join(dll_source, dll)
        dst = os.path.join(COMMON_LIB, dll)
        src_sz = os.path.getsize(src)
        dst_sz = os.path.getsize(dst) if os.path.exists(dst) else -1
        if src_sz == dst_sz:
            print(f"  {dll}: up-to-date ✓")
            skipped += 1
        else:
            shutil.copy2(src, dst)
            print(f"  {dll}: DEPLOYED ✅")
            deployed += 1
    
    # Step 3: Deploy to each terminal
    print(f"\n--- Per-terminal MQL5/Libraries ---")
    for entry in sorted(os.listdir(MT5_BASE)):
        if len(entry) != 32 or not entry.isalnum():
            continue
        lib_dir = os.path.join(MT5_BASE, entry, "MQL5", "Libraries")
        if not os.path.isdir(lib_dir):
            continue
        
        print(f"\n  Terminal {entry[:8]}…")
        for dll in REQUIRED_DLLS:
            src = os.path.join(dll_source, dll)
            dst = os.path.join(lib_dir, dll)
            src_sz = os.path.getsize(src)
            dst_sz = os.path.getsize(dst) if os.path.exists(dst) else -1
            if src_sz == dst_sz:
                print(f"    {dll}: up-to-date ✓")
                skipped += 1
            else:
                shutil.copy2(src, dst)
                print(f"    {dll}: DEPLOYED ✅")
                deployed += 1
    
    print(f"\n{'='*50}")
    print(f"  Deployed: {deployed}")
    print(f"  Skipped (up-to-date): {skipped}")
    print(f"{'='*50}")